In [31]:
from datetime import datetime, timedelta, timezone
import github3
import numpy
import os
from pathlib import Path
import sys
import time

In [2]:
REPO = 'bytecodealliance/wasm-micro-runtime'

*Please follow https://docs.github.com/en/authentication/keeping-your-account-and-data-secure/managing-your-personal-access-tokens to get a proper token.*

In [3]:
TOKEN_PATH = Path('./token')


In [4]:
PROXIES = {
    "http": "http://childprc-intel.com:913",
    "https": "http://childprc-intel.com:913"
}

In [5]:

def get_token(token_path):
    """
    Retrieve the GitHub token from a file or prompt the user to create it.
    """
    assert token_path.exists(), f"Token file not found. Please acquire a token, and store it at {token_path}"

    # if token is empty
    with open(token_path, 'r') as f:
        token = f.read().strip()
        assert token, f"Token file is empty. Please acquire a token, and store it at {token_path}"

        return token

In [6]:
def print_error_messages(error: github3.exceptions):
    """Prints the error messages from the GitHub API response.

    Args:
        Error (github3.exceptions): The error object from the GitHub API response.

    """
    if hasattr(error, "errors"):
        for e in error.errors:
            print(f"Error: {e.get('message')}")

In [7]:
def setup_connection(repo_full_name, token_path):
    """
    Make a connection to the specified GitHub repository.
    """
    token = get_token(token_path)
    gh = github3.GitHub(token=token)
    gh.session.proxies['http://'] = 'http://child-prc.intel.com:913'
    gh.session.proxies['https://'] = 'http://child-prc.intel.com:913'

    # a smoke test on connection
    current_user = gh.me()
    assert current_user.name == "liang.he", f"Expected user 'liang.he', but got '{current_user.name}'. Might want to change the token"

    return gh


In [8]:
def search_issues(github_connection, repos_and_owners_string, search_query, rate_limit_bypass = False):
    # Rate Limit Handling: API only allows 30 requests per minute
    def wait_for_api_refresh(
        iterator: github3.structs.SearchIterator, rate_limit_bypass: bool = False
    ):
        # If the rate limit bypass is enabled, don't wait for the API to refresh
        if rate_limit_bypass:
            return

        max_retries = 5
        retry_count = 0
        sleep_time = 70

        while iterator.ratelimit_remaining < 5:
            if retry_count >= max_retries:
                raise RuntimeError("Exceeded maximum retries for API rate limit")

            print(
                f"GitHub API Rate Limit Low, waiting {sleep_time} seconds to refresh."
            )
            time.sleep(sleep_time)

            # Exponentially increase the sleep time for the next retry
            sleep_time *= 2
            retry_count += 1

    print(f"Searching for issues... with {search_query} in {repos_and_owners_string}")
    issues_per_page = 100
    issues_iterator = github_connection.search_issues(
        search_query, per_page=issues_per_page
    )
    wait_for_api_refresh(issues_iterator, rate_limit_bypass)

    issues = []
    # Print the issue titles and add them to the list of issues
    try:
        for idx, issue in enumerate(issues_iterator, 1):
            print(issue.title)  # type: ignore
            issues.append(issue)

            # requests are sent once per page of issues
            if idx % issues_per_page == 0:
                wait_for_api_refresh(issues_iterator, rate_limit_bypass)

    except github3.exceptions.ForbiddenError as e:
        print(
            f"You do not have permission to view a repository \
from: '{repos_and_owners_string}'; Check your API Token."
        )
        print_error_messages(e)
        sys.exit(1)
    except github3.exceptions.NotFoundError as e:
        print(
            f"The repository could not be found; \
Check the repository owner and names: '{repos_and_owners_string}"
        )
        print_error_messages(e)
        sys.exit(1)
    except github3.exceptions.ConnectionError as e:
        print(
            "There was a connection error; Check your internet connection or API Token."
        )
        print_error_messages(e)
        sys.exit(1)
    except github3.exceptions.AuthenticationFailed as e:
        print("Authentication failed; Check your API Token.")
        print_error_messages(e)
        sys.exit(1)
    except github3.exceptions.UnprocessableEntity as e:
        print("The search query is invalid; Check the search query.")
        print_error_messages(e)
        sys.exit(1)

    return issues

def search_issues_created_in_last_90days(github_connection, repos_and_owners_string, search_query, rate_limit_bypass = False):
    """
    Search for issues in the specified GitHub repository within the last 90 days.

    Args:
        github_connection: The authenticated GitHub connection.

    Returns:
        list: A list of issues matching the search query.
    """
    now = datetime.now(timezone.utc)
    since = now - timedelta(days=90)
    search_query = f"{search_query} created:>={since.isoformat()}"

    return search_issues(github_connection, repos_and_owners_string, search_query, rate_limit_bypass)

In [38]:
def measure_time_to_close(issue):
    # Measure the time taken to close each issue
    if issue.state != "closed":
        print(f"Issue {issue.number} is not closed yet.")
        return None

    # removing tailing Z
    closed_at = datetime.fromisoformat(issue.closed_at[:-1])
    created_at = datetime.fromisoformat(issue.created_at[:-1])
    return closed_at - created_at

def get_stats_time_to_close(issues_with_data):
    if not issues_with_data or len(issues_with_data) == 0:
        print("No closed issues to analyze.")
        return None

    close_times = [t.total_seconds() for t in issues_with_data]

    average_time_to_close = numpy.round(numpy.average(close_times))
    med_time_to_close = numpy.round(numpy.median(close_times))
    ninety_percentile_time_to_close = numpy.round(
        numpy.percentile(close_times, 90, axis=0)
    )
    
    stats = {
        "avg": timedelta(seconds=average_time_to_close),
        "med": timedelta(seconds=med_time_to_close),
        "90p": timedelta(seconds=ninety_percentile_time_to_close),
    }

    # Print the average time to close converting seconds to a readable time format
    print(f"Time to close: {timedelta(seconds=average_time_to_close)}")
    return stats

In [ ]:
gh = setup_connection(REPO, TOKEN_PATH)
issues = search_issues_created_in_last_90days(gh, REPO, f"repo:{REPO} is:issue", False)


In [39]:
issues_time_to_close = []
for issue in issues:
    time_to_close = measure_time_to_close(issue)
    if time_to_close is None:
        continue

    print(f"Issue {issue.number} closed in {time_to_close}")
    issues_time_to_close.append(time_to_close)

get_stats_time_to_close(issues_time_to_close)


Issue 4561 is not closed yet.
Issue 4560 is not closed yet.
Issue 4558 is not closed yet.
Issue 4557 is not closed yet.
Issue 4544 is not closed yet.
Issue 4542 is not closed yet.
Issue 4539 is not closed yet.
Issue 4533 is not closed yet.
Issue 4528 is not closed yet.
Issue 4527 is not closed yet.
Issue 4525 is not closed yet.
Issue 4524 closed in 19:24:14
Issue 4523 is not closed yet.
Issue 4516 is not closed yet.
Issue 4506 is not closed yet.
Issue 4505 is not closed yet.
Issue 4504 closed in 25 days, 1:44:59
Issue 4501 is not closed yet.
Issue 4497 is not closed yet.
Issue 4495 is not closed yet.
Issue 4487 is not closed yet.
Issue 4484 is not closed yet.
Issue 4482 closed in 20 days, 3:15:46
Issue 4481 is not closed yet.
Issue 4480 is not closed yet.
Issue 4479 is not closed yet.
Issue 4478 closed in 13 days, 16:59:26
Issue 4475 is not closed yet.
Issue 4474 closed in 33 days, 0:08:36
Issue 4473 is not closed yet.
Issue 4464 closed in 34 days, 23:09:02
Issue 4463 is not closed yet

{'avg': datetime.timedelta(days=19, seconds=23814),
 'med': datetime.timedelta(days=11, seconds=78288),
 '90p': datetime.timedelta(days=48, seconds=4845)}